# Planning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/notebooks/intro/09_planning.ipynb)

Official API intro to `PlanningProblem` and trajectory optimization. Spatial
scenes, search (RRT), and DP policy synthesis have dedicated script demos.

**Scripts for depth:** `examples/scripts/trajectory_optimization/`, `examples/scripts/planning/`

**Heavier labs:** [`applications/car_trajopt.ipynb`](../applications/car_trajopt.ipynb) · [`applications/mpc.ipynb`](../applications/mpc.ipynb)


In [ ]:
# Local conda: minilink already installed. Colab: clone + path + meshcat.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q meshcat")


## `PlanningProblem` + direct collocation

Define a system, bounds, cost, and endpoints — then solve with
`TrajectoryOptimizationPlanner`.


In [ ]:
import numpy as np
from minilink.core.costs import QuadraticCost
from minilink.dynamics.catalog.pendulum.cartpole import CartPole
from minilink.planning.problems import PlanningProblem
from minilink.planning.trajectory_optimization.planner import TrajectoryOptimizationPlanner

sys = CartPole()
sys.inputs["u"].lower_bound[0] = -10.0
sys.inputs["u"].upper_bound[0] = 10.0

x_start = np.array([-1.0, 0.2, 0.0, 0.0])
x_goal = np.array([0.0, np.pi, 0.0, 0.0])

cost = QuadraticCost.from_system(
    sys,
    Q=np.diag([1.0, 1.0, 0.0, 0.0]),
    R=np.diag([1.0]),
    S=np.zeros((sys.n, sys.n)),
    xbar=x_goal,
    ubar=np.zeros(sys.m),
)
problem = PlanningProblem(sys=sys, tf=4.0, x_start=x_start, x_goal=x_goal, cost=cost)
planner = TrajectoryOptimizationPlanner(
    problem,
    n_steps=30,
    transcription="direct_collocation",
    solve_disp=False,
    optimizer_options={"disp": False, "maxiter": 200, "ftol": 1e-2},
)
result = planner.solve()
traj = result.trajectory
print("success:", getattr(result, "success", None), "cost:", getattr(result, "cost", None))
planner.problem.sys.plot_trajectory(traj)


## Spatial, search, and DP

- **Spatial / soft costs / scenes** — used heavily in MPC apps; see `minilink.planning.spatial` and `examples/scripts/mpc/`
- **RRT / RRT*** — `examples/scripts/planning/rrt/`
- **Value iteration / DP** — `examples/scripts/planning/value_iteration/`
